# Edit Tagger — Test Evaluation

This notebook evaluates the trained GEC edit tagger model on the QALB-2014-L1 test set.

**Pipeline:**
1. Process test `.sent`/`.cor` through the feature pipeline to get gold labels
2. Load the trained model checkpoint
3. Run inference on test examples
4. Compute token-level metrics (accuracy, precision, recall, F0.5)

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

## 1. Configuration

In [6]:
from src.services.gec.config import (
    LABEL2ID_PATH,
    MIN_LABEL_FREQUENCY,
    PROCESSED_DATA_DIR,
)

TEST_SENT_PATH = PROCESSED_DATA_DIR.parent / "raw" / "test" / "QALB-2014-L1-Test.sent"
TEST_COR_PATH = PROCESSED_DATA_DIR.parent / "raw" / "test" / "QALB-2014-L1-Test.cor"
TEST_JSONL_PATH = PROCESSED_DATA_DIR / "test_tokens_labels.jsonl"

MODEL_CHECKPOINT = Path("./gec_models/edit_tagger_v1/checkpoint-3642")
BASE_CHECKPOINT = "aubmindlab/bert-base-arabertv02"
MAX_LENGTH = 256
BATCH_SIZE = 32

## 2. Build test examples (feature pipeline)

Run the raw `.sent`/`.cor` files through alignment, projection, and compression to produce `ProjectedExample` objects with gold labels. Checkpoints are saved incrementally so you can resume if it crashes.

In [3]:
from src.services.gec.features.common import build_feature_builder
from src.services.gec.features.pruner import LabelPruner

FORCE_REBUILD = False

if FORCE_REBUILD or not TEST_JSONL_PATH.exists():
    builder = build_feature_builder()
    test_examples = builder.build_pipeline(
        TEST_SENT_PATH,
        TEST_COR_PATH,
        TEST_JSONL_PATH,
    )
    print(f"Built {len(test_examples)} test examples")

    pruner = LabelPruner(min_frequency=MIN_LABEL_FREQUENCY)
    test_examples = pruner.prune(test_examples)
    print(f"After pruning: {len(test_examples)} test examples")
else:
    print(f"Test JSONL already exists at {TEST_JSONL_PATH}")

/home/somia/baligh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


## 3. Load test dataset & vocabularies

In [ ]:
import json

from src.services.gec.training.datasets import GECTrainingDataset
from transformers import AutoTokenizer

with open(LABEL2ID_PATH, encoding="utf-8") as f:
    label2id = json.load(f)

id2label = {v: k for k, v in label2id.items()}
print(f"Label vocabulary: {len(label2id)} labels")

tokenizer = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)

test_dataset = GECTrainingDataset(
    jsonl_path=str(TEST_JSONL_PATH),
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)
print(f"Test examples: {len(test_dataset)}")

Label vocabulary: 3943 labels
Test examples: 968


## 4. Load trained model

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(MODEL_CHECKPOINT)
model.eval()
print(f"Model loaded from {MODEL_CHECKPOINT}")
print(f"num_labels = {model.config.num_labels}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 224.87it/s]


Model loaded from gec_models/edit_tagger_v1/checkpoint-3642
num_labels = 3943


In [8]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Device: {device}")

Device: cuda


## 5. Run inference

In [9]:
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
    shuffle=False,
)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()

        all_preds.append(pred_ids)
        all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f"Predictions shape: {all_preds.shape}")
print(f"Labels shape: {all_labels.shape}")

Evaluating: 100%|██████████| 31/31 [00:25<00:00,  1.23it/s]


Predictions shape: (968, 256)
Labels shape: (968, 256)


## 6. Compute overall metrics

In [10]:
IGNORE_INDEX = -100

mask = all_labels != IGNORE_INDEX
filtered_preds = all_preds[mask]
filtered_labels = all_labels[mask]

correct = int((filtered_preds == filtered_labels).sum())
total = len(filtered_labels)
accuracy = correct / total if total > 0 else 0.0

tp = fp = fn = 0
for pred, gold in zip(filtered_preds.tolist(), filtered_labels.tolist(), strict=False):
    if pred == gold:
        tp += 1
    else:
        fp += 1
        fn += 1

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
beta_sq = 0.25
f05 = (
    (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)
    if (beta_sq * precision + recall) > 0
    else 0.0
)

print("=" * 50)
print(f"{'Metric':<20} {'Value':>10}")
print("-" * 50)
print(f"{'Total tokens':<20} {total:>10}")
print(f"{'Correct':<20} {correct:>10}")
print(f"{'Accuracy':<20} {accuracy:>10.4f}")
print(f"{'Precision':<20} {precision:>10.4f}")
print(f"{'Recall':<20} {recall:>10.4f}")
print(f"{'F0.5':<20} {f05:>10.4f}")
print("=" * 50)

Metric                    Value
--------------------------------------------------
Total tokens              66670
Correct                   52285
Accuracy                 0.7842
Precision                0.7842
Recall                   0.7842
F0.5                     0.7842


## 7. Per-label breakdown

Metrics for each label type (sorted by support, top 50 shown).

In [ ]:
from collections import Counter

label_tp = Counter()
label_fp = Counter()
label_fn = Counter()
label_support = Counter()

for pred, gold in zip(filtered_preds.tolist(), filtered_labels.tolist(), strict=False):
    gold_name = id2label.get(gold, "[UNKNOWN]")
    pred_name = id2label.get(pred, "[UNKNOWN]")
    label_support[gold_name] += 1

    if pred == gold:
        label_tp[gold_name] += 1
    else:
        label_fp[pred_name] += 1
        label_fn[gold_name] += 1

rows = []
for label in sorted(
    label_support.keys(), key=lambda lab_name: label_support[lab_name], reverse=True
):
    tp = label_tp[label]
    fp = label_fp.get(label, 0)
    fn = label_fn.get(label, 0)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = (1 + beta_sq) * p * r / (beta_sq * p + r) if (beta_sq * p + r) > 0 else 0.0
    rows.append((label, label_support[label], p, r, f))

print(f"{'Label':<40} {'Support':>8} {'Prec':>8} {'Rec':>8} {'F0.5':>8}")
print("-" * 80)
for label, support, p, r, f in rows[:50]:
    print(f"{label:<40} {support:>8} {p:>8.4f} {r:>8.4f} {f:>8.4f}")
if len(rows) > 50:
    print(f"\n... and {len(rows) - 50} more labels")

Label                                     Support     Prec      Rec     F0.5
--------------------------------------------------------------------------------
K*                                          35132   0.9248   0.9858   0.9364
K                                            4103   0.8543   0.9517   0.8722
R_[،]                                        3295   0.7832   0.8704   0.7992
R_[c*]                                       1879   0.4040   0.6056   0.4328
D*                                           1363   0.5331   0.7982   0.5710
D*R_[c*]                                     1057   0.3413   0.6112   0.3743
R_[أ]K*                                      1044   0.9060   0.9598   0.9162
I_[c*]R_[c*]                                  979   0.3421   0.7140   0.3819
R_[.]                                         922   0.6699   0.6714   0.6702
DR_[c*]                                       911   0.3941   0.3392   0.3818
[UNK_EDIT]                                    800   0.2024   0.1250   0.

## 8. Operation-type macro metrics

Aggregate metrics by operation type: K (keep), R (replace), I (insert), D (delete).

In [ ]:
op_tp = Counter()
op_fp = Counter()
op_fn = Counter()
op_support = Counter()


def get_op(label_name: str) -> str:
    """Gets operation."""
    if (
        label_name.startswith("K")
        or label_name.startswith("[PAD")
        or label_name.startswith("[UNK")
    ):
        return "KEEP"
    if label_name.startswith("R_"):
        return "REPLACE"
    if label_name.startswith("I_"):
        return "INSERT"
    if label_name.startswith("D"):
        return "DELETE"
    return "OTHER"


for pred, gold in zip(filtered_preds.tolist(), filtered_labels.tolist(), strict=False):
    gold_name = id2label.get(gold, "[UNKNOWN]")
    pred_name = id2label.get(pred, "[UNKNOWN]")
    gold_op = get_op(gold_name)
    pred_op = get_op(pred_name)

    op_support[gold_op] += 1
    if gold_op == pred_op:
        op_tp[gold_op] += 1
    else:
        op_fp[pred_op] += 1
        op_fn[gold_op] += 1

print(f"{'Operation':<12} {'Support':>8} {'Prec':>8} {'Rec':>8} {'F0.5':>8}")
print("-" * 56)
for op in ["KEEP", "REPLACE", "INSERT", "DELETE", "OTHER"]:
    tp = op_tp.get(op, 0)
    fp = op_fp.get(op, 0)
    fn = op_fn.get(op, 0)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = (1 + beta_sq) * p * r / (beta_sq * p + r) if (beta_sq * p + r) > 0 else 0.0
    print(f"{op:<12} {op_support.get(op, 0):>8} {p:>8.4f} {r:>8.4f} {f:>8.4f}")

Operation     Support     Prec      Rec     F0.5
--------------------------------------------------------
KEEP            43644   0.9338   0.9603   0.9390
REPLACE         11306   0.8031   0.8022   0.8029
INSERT           4784   0.7986   0.6342   0.7593
DELETE           6933   0.7174   0.6928   0.7123
OTHER               3   0.0000   0.0000   0.0000


## 9. Sample predictions

In [ ]:
import json as json_mod

with TEST_JSONL_PATH.open(encoding="utf-8") as f:
    test_raw = [json_mod.loads(line) for line in f if line.strip()]

num_samples = 5

for i in range(min(num_samples, len(test_raw))):
    subwords = test_raw[i]["subwords"]
    gold_labels = test_raw[i].get("labels_star", test_raw[i].get("labels", []))

    pred_ids_i = all_preds[i][: len(subwords)]
    pred_names = [id2label.get(pid, "[UNK]") for pid in pred_ids_i]

    print(f"\n--- Example {i + 1} ---")
    mismatches = 0
    total_toks = 0
    for sw, gl, pl in zip(subwords, gold_labels, pred_names, strict=False):  # noqa: B007
        if gl != pl:
            mismatches += 1
        total_toks += 1
    token_acc = 1 - mismatches / total_toks
    correct_toks = total_toks - mismatches
    print(f"Token accuracy: {token_acc:.4f} ({correct_toks}/{total_toks})")
    print()
    print(f"{'Subword':<20} {'Gold':<30} {'Pred':<30} {'Match'}")
    print("-" * 80)
    for _sw, gl, pl in zip(subwords, gold_labels, pred_names, strict=False):
        mark = "OK" if gl == pl else "WRONG"
        if gl != pl:
            print(f"{sw:<20} {gl:<30} {pl:<30} {mark}")
    print("\n(Showing only mismatched tokens above)")


--- Example 1 ---
Token accuracy: 0.4198 (34/81)

Subword              Gold                           Pred                           Match
--------------------------------------------------------------------------------
##ه                  R_[ة]                          K*                             WRONG
يظن                  K*                             R_[ة]                          WRONG
ان                   R_[أ]K                         K*                             WRONG
ارواح                R_[أ]K*                        R_[أ]K                         WRONG
وآلام                K*                             R_[أ]K*                        WRONG
اقل                  R_[أ]K*                        K*                             WRONG
كلفه                 K*R_[ة]                        R_[أ]K*                        WRONG
من                   K*                             K*R_[ة]                        WRONG
##ه                  K                              K*             